#### Import Library

In [2]:
import pandas as pd
import re
import string
import joblib
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Acern\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acern\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

#### Load Dataset

In [3]:
df = pd.read_csv(
    '../../data/processed/cleaned_all_job.csv'
)

df.head()

,job_id,category,job_title,job_description,job_skill_set
0,3902668440,hr,sr human resource generalist,summary the sr hr generalist provides hr exper...,"['employee relations', 'talent acquisition', '..."
1,3905823748,hr,human resources manager,be part of a stellar team at ysb as the manage...,"['talent acquisition', 'employee performance m..."
2,3905854799,hr,director of human resources,our client is a thriving organization offering...,"['human resources management', 'recruitment', ..."
3,3905834061,hr,chief human resources officer,job title chief human resources officer chroin...,"['talent management', 'organizational developm..."
4,3906250451,hr,human resources generalist (hybrid role),description who we are avispl is a digital ena...,"['microsoft office', 'data analysis', 'employe..."


#### Create Model A

In [5]:
df['baseline_features'] = (
    df['category'].fillna('') + ' ' +
    df['job_title'].fillna('') + ' ' +
    df['job_description'].fillna('') + ' ' +
    df['job_skill_set'].fillna('')
)

In [6]:
def simple_preprocess(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r'\d+', '', text)

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    return text

In [7]:
df['baseline_text'] = (
    df['baseline_features']
    .apply(simple_preprocess)
)

#### TF-IDF Vectorization

In [8]:
tfidf_A = TfidfVectorizer(max_features=5000)

tfidf_matrix_A = tfidf_A.fit_transform(
    df['baseline_text']
)

print("TF-IDF Matrix Model A:")
print(tfidf_matrix_A.shape)

TF-IDF Matrix Model A:
(1167, 5000)


#### Hitung Cosine Similarity

In [10]:
cosine_sim_A = cosine_similarity(
    tfidf_matrix_A
)

print("Cosine Similarity Model A:")
print(cosine_sim_A.shape)

Cosine Similarity Model A:
(1167, 1167)


#### Load Model B

In [11]:
tfidf_matrix_B = joblib.load(
    '../../data/processed/tfidf_matrix.pkl'
)

cosine_sim_B = joblib.load(
    '../../data/processed/cosine_similarity.pkl'
)

tfidf_vectorizer_B = joblib.load(
    '../../data/processed/tfidf_vectorizer.pkl'
)

print("Model B berhasil dimuat")

Model B berhasil dimuat


c:\Users\Acern\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Acern\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [12]:
def get_recommendations(
    title,
    cosine_sim,
    df,
    top_n=5
):

    idx = df[
        df['job_title'] == title
    ].index[0]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n+1]

    job_indices = [
        i[0] for i in sim_scores
    ]

    return df[
        ['job_title']
    ].iloc[job_indices]

#### Testing

In [13]:
sample_job = df['job_title'].iloc[0]

print("Sample Job:")
print(sample_job)

Sample Job:
sr human resource generalist


In [14]:
print("=== Recommendation Model A ===")

get_recommendations(
    sample_job,
    cosine_sim_A,
    df
)

=== Recommendation Model A ===


,job_title
99,human resources director
96,human resources generalist
48,human resources manager
101,human resources generalist
150,human resources business partner


In [15]:
print("=== Recommendation Model B ===")

get_recommendations(
    sample_job,
    cosine_sim_B,
    df
)

=== Recommendation Model B ===


,job_title
48,human resources manager
99,human resources director
96,human resources generalist
150,human resources business partner
21,human resources generalist


#### Evaluasi

In [16]:
avg_similarity_A = cosine_sim_A.mean()

avg_similarity_B = cosine_sim_B.mean()

print("Average Similarity Model A:")
print(avg_similarity_A)

print("\nAverage Similarity Model B:")
print(avg_similarity_B)

Average Similarity Model A:
0.25268061928302626

Average Similarity Model B:
0.11358259282819377


#### Interpretasi Hasil A/B Testing

Model A memiliki rata-rata similarity lebih tinggi.

Hal ini menunjukkan bahwa Model A cenderung
menghasilkan hubungan antar pekerjaan yang lebih umum.

Sementara itu, Model B menggunakan preprocessing
lebih lengkap seperti:
- stopword removal
- tokenizing
- stemming

Preprocessing tersebut membuat sistem rekomendasi
lebih selektif sehingga similarity score menjadi
lebih rendah namun lebih spesifik.